### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="covertype",
    dataset_year="1998",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C50K5N",
    download_description="""
We download the data from UCI.

wget https://archive.ics.uci.edu/static/public/31/covertype.zip && unzip covertype.zip && rm old_covtype.info covtype.info covertype.zip  && gunzip covtype.data.gz
mkdir -p local-data-warehouse/covertype && mv covtype.data local-data-warehouse/covertype/
""",
    # References
    academic_reference_bibtex="""@article{blackard1999comparative,
  title={Comparative accuracies of artificial neural networks and discriminant analysis in predicting forest cover types from cartographic variables},
  author={Blackard, Jock A and Dean, Denis J},
  journal={Computers and electronics in agriculture},
  volume={24},
  number={3},
  pages={131--151},
  year={1999},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="blackard1999comparative",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped", "Spatial"],
    curation_comments="""
We start with the data from UCI.

We create a special version of the dataset to avoid leakage. This version consists only of three 3 classes and 3 (spatial) wilderness areas. The investigation of Covertype shows that the dataset comprises multiple wilderness areas (Rawah, Comanche Peak, Neota, Cache la Poudre) that can be treated as subgroups but are not strictly IID, with wilderness area and soil type encoded as one-hot categorical features. Although TabRed argues for a time split to reflect a real task, the dataset lacks both explicit time and precise spatial (e.g., GNSS) features, leaving only area identifiers, which implicitly encode collection time and location; consequently, IID splits would introduce temporal or spatial leakage. Additional issues include transformed features in the OpenML/TALENT version that leak test distribution, incorrect column order in the UCI release, and evidence from EDA that class distributions differ across areas, confirming the dataset’s grouped nature where cover type characteristics strongly depend on area. The only robust strategy is a spatially motivated grouped split by area; however, because some classes appear only in specific areas, the dataset is restricted to classes present in the three largest areas and evaluated using leave-one-area-out grouped splits.

Other steps:
- We add the column names in the correct way.
- We reverse the one-hot encoding of the Wilderness_Area and Soil_Type features, and add human-readable descriptions for these features. Moreover, we add the climatic and geologic zones categorical features for each soil type based on the codes.
- We reverse the ordinal encoding of the class name.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Cover_Type",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Cover_Type",
    group_on="Wilderness_Area",
    group_labels="per_sample",
)

## Preprocessing

In [2]:
import pandas as pd
import re

columns = ["Elevation","Aspect","Slope","Horizontal_Distance_To_Hydrology","Vertical_Distance_To_Hydrology",
           "Horizontal_Distance_To_Roadways","Hillshade_9am","Hillshade_Noon","Hillshade_3pm",
           "Horizontal_Distance_To_Fire_Points",
           "Wilderness_Area1", "Wilderness_Area2","Wilderness_Area3","Wilderness_Area4",
           "Soil_Type1","Soil_Type2","Soil_Type3","Soil_Type4", "Soil_Type5","Soil_Type6","Soil_Type7","Soil_Type8",
           "Soil_Type9","Soil_Type10","Soil_Type11","Soil_Type12", "Soil_Type13","Soil_Type14","Soil_Type15",
           "Soil_Type16","Soil_Type17","Soil_Type18","Soil_Type19", "Soil_Type20","Soil_Type21","Soil_Type22",
           "Soil_Type23","Soil_Type24","Soil_Type25","Soil_Type26", "Soil_Type27","Soil_Type28","Soil_Type29",
           "Soil_Type30","Soil_Type31","Soil_Type32","Soil_Type33", "Soil_Type34","Soil_Type35","Soil_Type36",
           "Soil_Type37","Soil_Type38","Soil_Type39","Soil_Type40",
           "Cover_Type",
           ]

df = pd.read_csv(dataset_mold.path / "covtype.data", header=None, names=columns)

print("Loaded data shape:", df.shape)
# --- Wilderness mapping (from "Wilderness_Area{k}" -> code + name) ---
WILDERNESS_NAME = {
    1: "Rawah Wilderness Area",
    2: "Neota Wilderness Area",
    3: "Comanche Peak Wilderness Area",
    4: "Cache la Poudre Wilderness Area",
}

# --- Soil mapping (Study Code -> USFS ELU code + description) ---
SOIL = {
  1:(2702,"Cathedral family - Rock outcrop complex, extremely stony."),
  2:(2703,"Vanet - Ratake families complex, very stony."),
  3:(2704,"Haploborolis - Rock outcrop complex, rubbly."),
  4:(2705,"Ratake family - Rock outcrop complex, rubbly."),
  5:(2706,"Vanet family - Rock outcrop complex complex, rubbly."),
  6:(2717,"Vanet - Wetmore families - Rock outcrop complex, stony."),
  7:(3501,"Gothic family."),
  8:(3502,"Supervisor - Limber families complex."),
  9:(4201,"Troutville family, very stony."),
 10:(4703,"Bullwark - Catamount families - Rock outcrop complex, rubbly."),
 11:(4704,"Bullwark - Catamount families - Rock land complex, rubbly."),
 12:(4744,"Legault family - Rock land complex, stony."),
 13:(4758,"Catamount family - Rock land - Bullwark family complex, rubbly."),
 14:(5101,"Pachic Argiborolis - Aquolis complex."),
 15:(5151,"unspecified in the USFS Soil and ELU Survey."),
 16:(6101,"Cryaquolis - Cryoborolis complex."),
 17:(6102,"Gateview family - Cryaquolis complex."),
 18:(6731,"Rogert family, very stony."),
 19:(7101,"Typic Cryaquolis - Borohemists complex."),
 20:(7102,"Typic Cryaquepts - Typic Cryaquolls complex."),
 21:(7103,"Typic Cryaquolls - Leighcan family, till substratum complex."),
 22:(7201,"Leighcan family, till substratum, extremely bouldery."),
 23:(7202,"Leighcan family, till substratum - Typic Cryaquolls complex."),
 24:(7700,"Leighcan family, extremely stony."),
 25:(7701,"Leighcan family, warm, extremely stony."),
 26:(7702,"Granile - Catamount families complex, very stony."),
 27:(7709,"Leighcan family, warm - Rock outcrop complex, extremely stony."),
 28:(7710,"Leighcan family - Rock outcrop complex, extremely stony."),
 29:(7745,"Como - Legault families complex, extremely stony."),
 30:(7746,"Como family - Rock land - Legault family complex, extremely stony."),
 31:(7755,"Leighcan - Catamount families complex, extremely stony."),
 32:(7756,"Catamount family - Rock outcrop - Leighcan family complex, extremely stony."),
 33:(7757,"Leighcan - Catamount families - Rock outcrop complex, extremely stony."),
 34:(7790,"Cryorthents - Rock land complex, extremely stony."),
 35:(8703,"Cryumbrepts - Rock outcrop - Cryaquepts complex."),
 36:(8707,"Bross family - Rock land - Cryumbrepts complex, extremely stony."),
 37:(8708,"Rock outcrop - Cryumbrepts - Cryorthents complex, extremely stony."),
 38:(8771,"Leighcan - Moran families - Cryaquolls complex, extremely stony."),
 39:(8772,"Moran family - Cryorthents - Leighcan family complex, extremely stony."),
 40:(8776,"Moran family - Cryorthents - Rock land complex, extremely stony."),
}

CLIMATIC_ZONE = {
    1:"lower montane dry",
    2:"lower montane",
    3:"montane dry",
    4:"montane",
    5:"montane dry and montane",
    6:"montane and subalpine",
    7:"subalpine",
    8:"alpine",
}
GEOLOGIC_ZONE = {
    1:"alluvium",
    2:"glacial",
    3:"shale",
    4:"sandstone",
    5:"mixed sedimentary",
    6:"unspecified in the USFS ELU Survey",
    7:"igneous and metamorphic",
    8:"volcanic",
}

# --- Target mapping ---
COVER_NAME = {
    1:"Spruce/Fir",
    2:"Lodgepole Pine",
    3:"Ponderosa Pine",
    4:"Cottonwood/Willow",
    5:"Aspen",
    6:"Douglas-fir",
    7:"Krummholz",
}


def merge_onehot(df, pattern, out_col, drop=True):
    cols = [c for c in df.columns if re.fullmatch(pattern, c)]
    X = df[cols]

    assert cols, f"No columns match {pattern}"
    assert X.notna().all().all(), f"{out_col}: contains NaNs"
    assert X.isin([0, 1]).all().all(), f"{out_col}: non-binary values found"
    s = X.sum(1)
    assert (s == 1).all(), f"{out_col}: not one-hot (row sums != 1). Bad rows: {(s!=1).sum()}"

    df[out_col] = X.idxmax(1)   # name of the active column
    return df.drop(columns=cols) if drop else df

df = merge_onehot(df, r"Wilderness_Area\d+", "Wilderness_Area", drop=True)
df = merge_onehot(df, r"Soil_Type\d+",       "Soil_Type",       drop=True)

# Wilderness -> code + description
w_code = df["Wilderness_Area"].str.extract(r"(\d+)$").astype("Int64")[0]
df["Wilderness_Area"] = w_code.map(WILDERNESS_NAME)

# Soil -> study code + ELU + description + climatic/geologic zones
s_code = df["Soil_Type"].str.extract(r"(\d+)$").astype("Int64")[0]
df["Soil_USFS_ELU"] = s_code.map(lambda k: SOIL[int(k)][0] if pd.notna(k) else pd.NA).astype("Int64")
df["Soil_Type"]     = s_code.map(lambda k: SOIL[int(k)][1] if pd.notna(k) else pd.NA)

elu = df["Soil_USFS_ELU"].astype("string")
df["Soil_ClimaticZone"] = elu.str[0].astype("Int64").map(CLIMATIC_ZONE)
df["Soil_GeologicZone"] = elu.str[1].astype("Int64").map(GEOLOGIC_ZONE)
df = df.drop(columns=["Soil_USFS_ELU"])

# Target -> text label
df["Cover_Type"] = df["Cover_Type"].map(COVER_NAME)

# Show data distribution across wilderness areas and cover types to confirm the grouped nature of the dataset
df.groupby(["Wilderness_Area", "Cover_Type"]).size().unstack(fill_value=0).sort_index()

Loaded data shape: (581012, 55)


Cover_Type,Aspen,Cottonwood/Willow,Douglas-fir,Krummholz,Lodgepole Pine,Ponderosa Pine,Spruce/Fir
Wilderness_Area,,,,,,,
Cache la Poudre Wilderness Area,0,2747,9741,0,3026,21454,0
Comanche Peak Wilderness Area,5712,0,7626,13105,125093,14300,87528
Neota Wilderness Area,0,0,0,2304,8985,0,18595
Rawah Wilderness Area,3781,0,0,5101,146197,0,105717


In [3]:
# Create special non-leakage version of covertype for our task

# Keep only data points that have the following classes and areas:
allowed_classes = ["Krummholz", "Lodgepole Pine", "Spruce/Fir"]
allowed_areas = ["Comanche Peak Wilderness Area", "Neota Wilderness Area", "Rawah Wilderness Area"]
df = df[df["Cover_Type"].isin(allowed_classes) & df["Wilderness_Area"].isin(allowed_areas)].reset_index(drop=True)

# Handle cats
as_cat_tye = ["Wilderness_Area", "Soil_Type", "Soil_ClimaticZone", "Soil_GeologicZone", "Cover_Type"]
df[as_cat_tye] = df[as_cat_tye].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 512,625
Columns: 15
Use sampling: False (sample size: 512,625)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Horizontal_Distance_To_Fire_Points', 'Horizontal_Distance_To_Roadways', 'Elevation', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Hydrology', 'Aspect', 'Hillshade_3pm', 'Hillshade_9am', 'Hillshade_Noon', 'Slope']
Rows remaining as candidates after top-10 filter: 0 (of 512,625)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type,Wilderness_Area,Soil_Type,Soil_ClimaticZone,Soil_GeologicZone
0,3276,347,23,212,49,1604,171,197,160,2780,Spruce/Fir,Rawah Wilderness Area,"Moran family - Cryorthents - Leighcan family complex, extremely stony.",alpine,igneous and metamorphic
1,2703,13,31,60,33,2985,173,159,108,6333,Lodgepole Pine,Rawah Wilderness Area,"Como - Legault families complex, extremely stony.",subalpine,igneous and metamorphic
2,3231,308,10,306,67,1805,191,234,182,2368,Spruce/Fir,Comanche Peak Wilderness Area,"Leighcan - Catamount families - Rock outcrop complex, extremely stony.",subalpine,igneous and metamorphic
3,3031,178,15,255,30,1482,225,248,150,534,Spruce/Fir,Comanche Peak Wilderness Area,"Catamount family - Rock land - Bullwark family complex, rubbly.",montane,igneous and metamorphic
4,3004,344,16,295,39,3375,187,214,165,2067,Spruce/Fir,Comanche Peak Wilderness Area,"Leighcan family, extremely stony.",subalpine,igneous and metamorphic


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Cover_Type,category,0.0,0.0,3.0,"Lodgepole Pine, Spruce/Fir, Krummholz"
1,Wilderness_Area,category,0.0,0.0,3.0,"Rawah Wilderness Area, Comanche Peak Wilderness Area, Neota Wilderness Area"
2,Soil_Type,category,0.0,0.0,35.0,"Como - Legault families complex, extremely stony., Leighcan family, till substratum - Typic Cryaquolls complex., Catamount family - Rock outcrop - Leighcan family complex, extremely stony., Leighcan - Catamount families - Rock outcrop complex, extremely stony., Leighcan family, till substratum, extremely bouldery., Legault family - Rock land complex, stony., Como family - Rock land - Legault family complex, extremely stony., Leighcan - Catamount families complex, extremely stony., Leighcan family, extremely stony., Leighcan - Moran families - Cryaquolls complex, extremely stony."
3,Soil_ClimaticZone,category,0.0,0.0,6.0,"subalpine, montane, alpine, lower montane, montane and subalpine, montane dry"
4,Soil_GeologicZone,category,0.0,0.0,4.0,"igneous and metamorphic, glacial, alluvium, mixed sedimentary"
5,Elevation,int64,0.0,0.0,1507.0,"2968, 2991, 2962, 2972, 2975, 2978, 2988, 2955, 2965, 2985"
6,Aspect,int64,0.0,0.0,361.0,"45, 0, 90, 135, 63, 315, 72, 27, 18, 36"
7,Slope,int64,0.0,0.0,67.0,"10, 11, 12, 9, 13, 8, 14, 15, 7, 16"
8,Horizontal_Distance_To_Hydrology,int64,0.0,0.0,551.0,"30, 0, 150, 60, 67, 42, 108, 85, 90, 120"
9,Vertical_Distance_To_Hydrology,int64,0.0,0.0,700.0,"0, 3, 10, 7, 6, 13, 4, 5, 16, 2"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Elevation,512625.0,3027.064808,207.581709,2313.0,3858.0
Aspect,512625.0,153.408298,111.402664,0.0,360.0
Slope,512625.0,13.384858,6.999166,0.0,66.0
Horizontal_Distance_To_Hydrology,512625.0,279.748024,217.892493,0.0,1397.0
Vertical_Distance_To_Hydrology,512625.0,45.247774,58.493457,-173.0,601.0
Horizontal_Distance_To_Roadways,512625.0,2529.546836,1555.458974,0.0,7117.0
Hillshade_9am,512625.0,213.348069,24.738071,0.0,254.0
Hillshade_Noon,512625.0,224.432156,18.446205,0.0,254.0
Hillshade_3pm,512625.0,142.905709,36.231254,0.0,254.0
Horizontal_Distance_To_Fire_Points,512625.0,2106.722150,1339.239592,0.0,7173.0


In [8]:
# Categorical Feature Statistics
cat_stats

value  \
column            rank                                                                        
Cover_Type        1                                                          Lodgepole Pine   
                  2                                                              Spruce/Fir   
                  3                                                               Krummholz   
Soil_ClimaticZone 1                                                               subalpine   
                  2                                                                 montane   
                  3                                                                  alpine   
                  4                                                           lower montane   
                  5                                                   montane and subalpine   
Soil_GeologicZone 1                                                 igneous and metamorphic   
                  2                                                                 glacial   
                  3                                                                alluvium   
                  4                                                       mixed sedimentary   
Soil_Type         1                       Como - Legault families complex, extremely stony.   
                  2            Leighcan family, till substratum - Typic Cryaquolls complex.   
                  3     Catamount family - Rock outcrop - Leighcan family complex, extre...   
                  4     Leighcan - Catamount families - Rock outcrop complex, extremely ...   
                  5                   Leighcan family, till substratum, extremely bouldery.   
Wilderness_Area   1                                                   Rawah Wilderness Area   
                  2                                           Comanche Peak Wilderness Area   
                  3                                                   Neota Wilderness Area   

                         count    pct  
column            rank                 
Cover_Type        1     280275  54.67  
                  2     211840  41.32  
                  3      20510   4.00  
Soil_ClimaticZone 1     395023  77.06  
                  2      66248  12.92  
                  3      40437   7.89  
                  4       5399   1.05  
                  5       5234   1.02  
Soil_GeologicZone 1     403576  78.73  
                  2      91544  17.86  
                  3      17221   3.36  
                  4        284   0.06  
Soil_Type         1     114115  22.26  
                  2      57024  11.12  
                  3      51753  10.10  
                  4      44092   8.60  
                  5      33373   6.51  
Wilderness_Area   1     257015  50.14  
                  2     225726  44.03  
                  3      29884   5.83

In [9]:
# Target Distribution
target_df

,count,pct
Cover_Type,,
Lodgepole Pine,280275,54.67
Spruce/Fir,211840,41.32
Krummholz,20510,4.00


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=1, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 3-fold split, always leaving one area out of the training data.",
    splits=splits
)

Using Stratified Grouped splits.
Using label-per-sample grouped splits.
Repeat 0, Fold 0:
            Train N: 286899, Test N: 225726
            Target Distribution:
            	Train target distribution: {'Lodgepole Pine': 0.5408941822732042, 'Spruce/Fir': 0.43329534086908633, 'Krummholz': 0.025810476857709506}
            	Test target distribution: {'Lodgepole Pine': 0.5541807323923695, 'Spruce/Fir': 0.3877621541160522, 'Krummholz': 0.058057113491578285}
            Group Distribution Wilderness_Area:
            	Train: 2
            	Test: 1
            
Repeat 0, Fold 1:
            Train N: 482741, Test N: 29884
            Target Distribution:
            	Train target distribution: {'Lodgepole Pine': 0.5619783693533386, 'Spruce/Fir': 0.4003078255213458, 'Krummholz': 0.03771380512531564}
            	Test target distribution: {'Spruce/Fir': 0.6222393253915138, 'Lodgepole Pine': 0.3006625619060367, 'Krummholz': 0.07709811270244947}
            Group Distribution Wilderness_Area

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to covertype/019d319b-4e2b-7eb3-b3ba-5082672806cb
019d319b-4e2b-7eb3-b3ba-5082672806cb
cacc32897328dfb1ac739bc69e20271601f476f745639ff60c9e6a68500611e5
